# Building Multi-Step Tool-Calling Datasets with Data Designer

## A Complete Tutorial for Generating Training Data for Agentic RL

This notebook teaches you how to build synthetic datasets for training multi-step tool-calling agents using NVIDIA's Data Designer. By the end, you'll understand:

1. **Environment Design**: How to structure tools and databases for agentic tasks
2. **Synthetic Data Generation**: Using LLMs to generate realistic user queries and tool trajectories
3. **Quality Assurance**: Using LLM judges to filter and audit generated data
4. **RL Integration**: Formatting data for NeMo Gym rollout collection and GRPO training

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         DATA GENERATION PIPELINE                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐              │
│  │ Tool Schemas │───▶│ User Query   │───▶│  Trajectory  │              │
│  │   (Seed)     │    │ Generation   │    │  Simulation  │              │
│  └──────────────┘    └──────────────┘    └──────────────┘              │
│                                                 │                        │
│                                                 ▼                        │
│                                          ┌──────────────┐              │
│                                          │  LLM Judge   │              │
│                                          │  (Quality)   │              │
│                                          └──────────────┘              │
│                                                 │                        │
│                                                 ▼                        │
│                                          ┌──────────────┐              │
│                                          │ NeMo Gym     │              │
│                                          │ Format       │              │
│                                          └──────────────┘              │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

## Part 1: Setup and Dependencies

First, let's install and import the necessary libraries.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install data-designer pydantic pandas

In [ ]:
import json
import random
from typing import List, Optional
from pydantic import BaseModel, Field
import pandas as pd

# Data Designer imports
import data_designer.essentials as dd
from data_designer.essentials import Score, SamplingStrategy

## Part 2: Load Tool Definitions

The **Workplace Assistant** environment has 26 tools across 5 databases:
- **Email**: Send, search, reply, forward, delete emails
- **Calendar**: Create, search, update, delete events
- **Analytics**: Query website visitor data and create plots
- **Project Management**: Manage tasks across Kanban boards
- **CRM**: Manage customer records and sales pipeline

These tools are designed to require **multi-step reasoning**. For example:
- "Email John about the meeting" → First lookup John's email, then send email
- "Reassign all of Sarah's leads to Mike" → Lookup emails, search customers, update each one

In [ ]:
# Load tool definitions from separate JSON files (one per database)
import os

TOOLS_DIR = 'tools'

# Load environment config
with open(os.path.join(TOOLS_DIR, 'environment.json'), 'r') as f:
    env_config = json.load(f)

SYSTEM_PROMPT = env_config['system_prompt']
MULTI_STEP_PATTERNS = env_config['common_multi_step_patterns']

# Load tools from each database file
DATABASE_FILES = [
    'company_directory.json',
    'email.json', 
    'calendar.json',
    'analytics.json',
    'project_management.json',
    'customer_relationship_manager.json'
]

TOOLS = []
DATABASES = {}
TOOL_CATEGORIES = {}

for db_file in DATABASE_FILES:
    with open(os.path.join(TOOLS_DIR, db_file), 'r') as f:
        db_config = json.load(f)
        
    db_name = db_config['database']
    DATABASES[db_name] = {
        'description': db_config['description'],
        'data_schema': db_config['data_schema']
    }
    
    # Add tools and track category
    db_tools = db_config['tools']
    TOOLS.extend(db_tools)
    TOOL_CATEGORIES[db_name] = [t['name'] for t in db_tools]

print(f"Loaded {len(TOOLS)} tools across {len(DATABASES)} databases")
print(f"\nDatabases:")
for db_name, db_info in DATABASES.items():
    tool_count = len(TOOL_CATEGORIES[db_name])
    print(f"  - {db_name}: {tool_count} tools")
    print(f"    {db_info['description']}")

In [ ]:
# Helper function to format tools for prompts
def format_tools_for_prompt(tools: List[dict], include_schemas: bool = False) -> str:
    """Format tool definitions into a readable string for LLM prompts."""
    lines = []
    for tool in tools:
        lines.append(f"- **{tool['name']}**: {tool['description']}")
        if include_schemas:
            params = tool['parameters']['properties']
            if params:
                lines.append(f"  Parameters: {list(params.keys())}")
    return "\n".join(lines)

# Display tool summary by category
for category, tool_names in TOOL_CATEGORIES.items():
    print(f"\n### {category.upper()} ({len(tool_names)} tools)")
    category_tools = [t for t in TOOLS if t['name'] in tool_names]
    print(format_tools_for_prompt(category_tools))

## Part 3: Define Output Schemas (Pydantic Models)

Data Designer uses Pydantic models to define structured output formats. This ensures the LLM generates data in a consistent, parseable format.

We define:
1. **ToolCall**: A single tool invocation with name and arguments
2. **AgentStep**: One step in a trajectory (thought + tool call + expected result)
3. **AgentTrajectory**: The complete multi-step solution

In [ ]:
class ToolCall(BaseModel):
    """A single tool invocation."""
    name: str = Field(..., description="The name of the tool to call (e.g., 'email_send_email')")
    arguments: str = Field(..., description="JSON string of the tool arguments")


class AgentStep(BaseModel):
    """A single step in the agent's reasoning trajectory."""
    step_number: int = Field(..., description="The step number (1-indexed)")
    thought: str = Field(
        ..., 
        description="The agent's reasoning about what to do next and why. Should explain the purpose of the tool call."
    )
    tool_call: ToolCall = Field(..., description="The tool to call in this step")
    expected_result: str = Field(
        ..., 
        description="What information or state change we expect from this tool call"
    )


class AgentTrajectory(BaseModel):
    """Complete trajectory for solving a multi-step task."""
    reasoning_trace: List[AgentStep] = Field(
        ..., 
        description="The sequence of steps to solve the task. Should be 1-6 steps."
    )
    final_answer: str = Field(
        ..., 
        description="A brief confirmation of what was accomplished"
    )


class JudgeScores(BaseModel):
    """Quality scores from the LLM judge."""
    validity: int = Field(..., ge=1, le=5, description="Are the tool calls valid and executable? (1=invalid, 5=perfect)")
    complexity: int = Field(..., ge=1, le=5, description="Does the task require multi-step reasoning? (1=trivial, 5=complex)")
    quality: int = Field(..., ge=1, le=5, description="Is the trajectory efficient and well-reasoned? (1=poor, 5=excellent)")
    reasoning: str = Field(..., description="Brief explanation of the scores")

## Part 4: Define Generation Prompts

The heart of synthetic data generation is the prompts. We need prompts for:

1. **User Query Generation**: Create realistic workplace requests
2. **Trajectory Simulation**: Generate the step-by-step solution
3. **Quality Judging**: Evaluate the generated data

### Key Principles:
- **Specificity**: Tell the LLM exactly what format you want
- **Examples**: Show don't tell - include concrete examples
- **Constraints**: Define what NOT to do (avoid trivial tasks, don't skip steps)

In [ ]:
# Prompt 1: Generate a realistic user query that requires multiple tool calls
USER_QUERY_GENERATION_PROMPT = """
You are creating training data for a workplace assistant AI agent.

**Your Task:** Generate a realistic user request that requires the agent to use multiple tools to complete.

**Available Tools:**
{{ tools_description }}

**Selected Tool Category:** {{ category }}

**Multi-Step Pattern to Use:** {{ pattern }}

**Guidelines:**
1. The request should sound natural - like something a real employee would ask
2. It MUST require 2-6 tool calls to complete (not just one!)
3. Include specific details that make the task concrete (names, dates, subjects)
4. Don't mention tool names or technical details - speak like a normal user
5. The task should be achievable with the available tools

**Good Examples:**
- "Raj is taking over all of Akira's leads that are interested in software. Can you reassign them in the CRM?"
- "Forward the last email from marketing about the Q4 report to everyone on the design team"
- "Move all of Sarah's overdue tasks on the Backend board to the Backlog"

**Bad Examples (too simple):**
- "Send an email to john@example.com" (only 1 tool call)
- "What's on my calendar tomorrow?" (just a search, no action)

**Output:** Return ONLY the user request as a single string. No quotes, no explanation.
"""

print("User Query Generation Prompt loaded")

In [ ]:
# Prompt 2: Simulate the agent's trajectory for solving the task
TRAJECTORY_SIMULATION_PROMPT = """
You are simulating an expert workplace assistant agent solving a task step-by-step.

**User Request:**
{{ user_query }}

**System Context:**
{{ system_prompt }}

**Available Tools:**
{{ tools_json }}

**Your Task:** Generate a step-by-step trajectory showing how the agent would solve this request.

**Guidelines:**
1. **Think Step-by-Step**: Each step should have a clear thought explaining WHY we're calling this tool
2. **Use Real Tool Names**: The tool_call.name must exactly match one of the available tools
3. **Valid JSON Arguments**: The tool_call.arguments must be valid JSON matching the tool's parameter schema
4. **Realistic IDs**: When referencing IDs discovered in previous steps, use placeholder format like "00000001"
5. **Complete the Task**: The trajectory must fully solve the user's request
6. **2-6 Steps**: Most tasks need 2-6 tool calls. Don't pad with unnecessary steps.

**Common Patterns:**
- Look up a person's email before sending them a message
- Search for records before updating/deleting them
- Get information from one database to use in another

**Example Step:**
```json
{{
  "step_number": 1,
  "thought": "The user wants to email Raj, but I need his email address first. I'll look it up in the company directory.",
  "tool_call": {{
    "name": "company_directory_find_email_address",
    "arguments": "{\\"name\\": \\"Raj\\"}"
  }},
  "expected_result": "Raj's email address (likely raj.patel@atlas.com)"
}}
```

Output the complete AgentTrajectory with all steps needed to solve the task.
"""

print("Trajectory Simulation Prompt loaded")

In [ ]:
# Prompt 3: Judge the quality of the generated data
JUDGE_PROMPT = """
You are a quality assurance judge evaluating synthetic training data for an AI agent.

**User Request:**
{{ user_query }}

**Generated Trajectory:**
{{ trajectory }}

**Available Tools:**
{{ tools_summary }}

**Evaluate on these criteria:**

1. **Validity (1-5)**: Are all tool calls valid?
   - Tool names must exactly match available tools
   - Arguments must match the tool's parameter schema
   - The sequence must be logically executable
   - Score 1 if any tool call is invalid, 5 if all are perfect

2. **Complexity (1-5)**: Does this require real multi-step reasoning?
   - Score 1-2 for tasks solvable in 1 step
   - Score 3 for 2-step tasks
   - Score 4-5 for tasks requiring 3+ steps with dependencies between them

3. **Quality (1-5)**: Is the trajectory efficient and well-reasoned?
   - Are thoughts clear and explanatory?
   - Is the solution optimal (no unnecessary steps)?
   - Does it fully solve the user's request?

**Output:** Return JudgeScores with ratings and brief reasoning.
"""

print("Judge Prompt loaded")

## Part 5: Create Seed Data

Data Designer works by expanding seed data through LLM generation. Our seeds contain:
- Tool category to focus on
- Multi-step pattern to use
- Formatted tool descriptions

By varying the seeds, we ensure diversity in the generated data.

In [ ]:
def create_seed_data(num_seeds: int = 100) -> pd.DataFrame:
    """
    Create seed data for the Data Designer pipeline.
    
    Each seed contains:
    - category: Which tool category to focus on
    - pattern: Which multi-step pattern to use
    - tools_description: Formatted tool descriptions
    - tools_json: Full tool schemas as JSON
    - system_prompt: The system context
    """
    seeds = []
    
    categories = list(TOOL_CATEGORIES.keys())
    patterns = [
        "lookup_then_action: Look up a person's email, then perform an action using that email",
        "search_then_batch_update: Search for records matching criteria, then update each one",
        "search_then_batch_delete: Search for records matching criteria, then delete each one",
        "cross_database: Query one database to get info needed for another database",
        "multi_search_then_action: Search multiple databases, combine results, then act",
    ]
    
    for i in range(num_seeds):
        # Select category and pattern (ensuring diversity)
        category = categories[i % len(categories)]
        pattern = patterns[i % len(patterns)]
        
        # Get tools for this category (plus company_directory for lookups)
        relevant_tool_names = TOOL_CATEGORIES[category] + TOOL_CATEGORIES.get('company_directory', [])
        relevant_tools = [t for t in TOOLS if t['name'] in relevant_tool_names]
        
        seeds.append({
            'seed_id': i,
            'category': category,
            'pattern': pattern,
            'tools_description': format_tools_for_prompt(relevant_tools, include_schemas=True),
            'tools_json': json.dumps(relevant_tools, indent=2),
            'tools_summary': format_tools_for_prompt(TOOLS),  # All tools for judge
            'system_prompt': SYSTEM_PROMPT,
        })
    
    return pd.DataFrame(seeds)

# Create seed data
seed_df = create_seed_data(num_seeds=50)
print(f"Created {len(seed_df)} seeds")
print(f"\nSample seed:")
print(seed_df.iloc[0].to_dict())

In [ ]:
# Save seeds to parquet for Data Designer
seed_df.to_parquet('workplace_assistant_seeds.parquet', index=False)
print("Seeds saved to workplace_assistant_seeds.parquet")

## Part 6: Configure the Data Designer Pipeline

Now we wire everything together into a Data Designer workflow:

1. **Load Seeds** → Provides category, pattern, tools for each generation
2. **Generate User Query** → LLM creates realistic request
3. **Simulate Trajectory** → LLM generates step-by-step solution
4. **Judge Quality** → LLM evaluates validity, complexity, quality

The output is a dataset ready for NeMo Gym rollout collection.

In [ ]:
# Model configuration
# For production, use a capable model like GPT-4 or Nemotron
MODEL_ALIAS = "generator"
MODEL_NAME = "openai/gpt-4o"  # Or your preferred model endpoint

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_NAME,
        inference_parameters=dd.InferenceParameters(
            temperature=0.7,  # Some creativity for diverse outputs
            max_tokens=4096,  # Enough for complex trajectories
            max_parallel_requests=32,
        ),
    ),
]

print(f"Using model: {MODEL_NAME}")

In [ ]:
def build_workplace_assistant_pipeline():
    """
    Build the complete Data Designer pipeline for generating 
    multi-step tool-calling training data.
    """
    
    # Initialize the config builder
    config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
    
    # Load seed data
    seed_ref = dd.DataDesigner.make_seed_reference_from_file(
        file_path='workplace_assistant_seeds.parquet'
    )
    config_builder.with_seed_dataset(seed_ref, sampling_strategy=SamplingStrategy.SHUFFLE)
    
    # Column 1: Generate User Query
    # This creates a realistic workplace request based on the category and pattern
    config_builder.add_column(
        dd.LLMTextColumnConfig(
            name="user_query",
            prompt=USER_QUERY_GENERATION_PROMPT,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 2: Simulate Agent Trajectory
    # This generates the step-by-step solution with tool calls
    config_builder.add_column(
        dd.LLMStructuredColumnConfig(
            name="trajectory",
            prompt=TRAJECTORY_SIMULATION_PROMPT,
            output_format=AgentTrajectory,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 3: Judge Quality
    # This evaluates the generated data for validity, complexity, and quality
    config_builder.add_column(
        dd.LLMStructuredColumnConfig(
            name="judge_scores",
            prompt=JUDGE_PROMPT,
            output_format=JudgeScores,
            model_alias=MODEL_ALIAS,
        )
    )
    
    return config_builder

# Build the pipeline
pipeline = build_workplace_assistant_pipeline()
print("Pipeline configured with 3 generation columns:")
print("  1. user_query (text)")
print("  2. trajectory (structured - AgentTrajectory)")
print("  3. judge_scores (structured - JudgeScores)")

## Part 7: Run the Pipeline (Local Testing)

For local testing, we'll run a small batch. For production, you'd use Big Iron to scale across GPUs.

In [ ]:
# For local testing with a small batch
# Uncomment to run:

# designer = dd.DataDesigner.from_config_builder(pipeline)
# results = designer.generate(num_records=5)
# results_df = results.to_pandas()
# print(results_df.head())

## Part 8: Convert to NeMo Gym Format

The generated data needs to be converted to the format expected by NeMo Gym for rollout collection. This format includes:

- `id`: Unique identifier
- `responses_create_params`: Input messages and tool schemas
- `ground_truth`: Expected tool calls
- `category`: Task category for stratification
- `environment_name`: Always "workplace_assistant"

In [ ]:
def convert_to_nemo_gym_format(row: dict, idx: int) -> dict:
    """
    Convert a generated row to NeMo Gym rollout format.
    
    This format is what NeMo Gym's workplace_assistant environment expects.
    The rollout collector will:
    1. Send the input messages to the model
    2. Collect the model's tool calls
    3. Execute them in the environment
    4. Compare final state to ground_truth execution
    5. Compute reward (1 if states match, 0 otherwise)
    """
    
    # Parse the trajectory
    trajectory = row.get('trajectory', {})
    if isinstance(trajectory, str):
        trajectory = json.loads(trajectory)
    
    # Extract tool calls for ground truth
    ground_truth = []
    for step in trajectory.get('reasoning_trace', []):
        tool_call = step.get('tool_call', {})
        ground_truth.append({
            'name': tool_call.get('name', ''),
            'arguments': tool_call.get('arguments', '{}')
        })
    
    # Build the responses_create_params (OpenAI-compatible format)
    responses_create_params = {
        'input': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row.get('user_query', '')}
        ],
        'tools': TOOLS,  # All 26 tools available
        'parallel_tool_calls': False,  # Sequential execution
        'temperature': 1.0,  # For rollout diversity
    }
    
    return {
        'id': idx,
        'responses_create_params': responses_create_params,
        'ground_truth': ground_truth,
        'category': f"workplace_assistant_{row.get('category', 'general')}",
        'environment_name': 'workplace_assistant',
        # Metadata for analysis (not used by NeMo Gym)
        'judge_scores': row.get('judge_scores', {}),
        'pattern': row.get('pattern', ''),
    }

print("Conversion function defined")

In [ ]:
def filter_high_quality(df: pd.DataFrame, min_validity: int = 4, min_complexity: int = 3) -> pd.DataFrame:
    """
    Filter generated data to keep only high-quality examples.
    
    This is crucial for training - low-quality examples can hurt model performance.
    
    Args:
        df: DataFrame with judge_scores column
        min_validity: Minimum validity score (1-5). Recommend 4+.
        min_complexity: Minimum complexity score (1-5). Recommend 3+.
    """
    
    def parse_scores(scores):
        if isinstance(scores, str):
            return json.loads(scores)
        return scores or {}
    
    # Parse judge scores
    df['_scores'] = df['judge_scores'].apply(parse_scores)
    
    # Filter
    mask = (
        (df['_scores'].apply(lambda x: x.get('validity', 0)) >= min_validity) &
        (df['_scores'].apply(lambda x: x.get('complexity', 0)) >= min_complexity)
    )
    
    filtered = df[mask].drop(columns=['_scores']).reset_index(drop=True)
    
    print(f"Filtered {len(df)} -> {len(filtered)} examples")
    print(f"  Kept {len(filtered)/len(df)*100:.1f}% of data")
    
    return filtered

print("Filter function defined")

In [ ]:
def save_for_nemo_gym(df: pd.DataFrame, output_path: str):
    """
    Save the dataset in JSONL format for NeMo Gym.
    
    Each line is a complete training example that can be used for:
    - Rollout collection (the model attempts to solve the task)
    - Reward computation (comparing model output to ground truth)
    - GRPO training (optimizing the policy using the rewards)
    """
    
    with open(output_path, 'w') as f:
        for idx, row in df.iterrows():
            record = convert_to_nemo_gym_format(row.to_dict(), idx)
            f.write(json.dumps(record) + '\n')
    
    print(f"Saved {len(df)} examples to {output_path}")

print("Save function defined")

## Part 9: Example Output

Let's create a mock example to show what the final output looks like.

In [ ]:
# Mock example showing the expected output format
example_output = {
    'id': 0,
    'responses_create_params': {
        'input': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': "Raj is taking over all of Akira's leads that are interested in software. Can you reassign them in the CRM?"}
        ],
        'tools': TOOLS,
        'parallel_tool_calls': False,
        'temperature': 1.0,
    },
    'ground_truth': [
        {'name': 'company_directory_find_email_address', 'arguments': '{"name": "Akira"}'},
        {'name': 'company_directory_find_email_address', 'arguments': '{"name": "Raj"}'},
        {'name': 'customer_relationship_manager_search_customers', 'arguments': '{"assigned_to_email": "akira.tanaka@atlas.com", "product_interest": "software", "status": "lead"}'},
        {'name': 'customer_relationship_manager_update_customer', 'arguments': '{"customer_id": "00000095", "field": "assigned_to_email", "new_value": "raj.patel@atlas.com"}'},
        {'name': 'customer_relationship_manager_update_customer', 'arguments': '{"customer_id": "00000080", "field": "assigned_to_email", "new_value": "raj.patel@atlas.com"}'},
    ],
    'category': 'workplace_assistant_customer_relationship_manager',
    'environment_name': 'workplace_assistant',
}

print("Example NeMo Gym format:")
print(json.dumps(example_output, indent=2)[:2000] + "...")

## Part 10: End-to-End Workflow Summary

Here's the complete workflow for generating training data:

```python
# 1. Create seeds
seed_df = create_seed_data(num_seeds=1000)
seed_df.to_parquet('seeds.parquet')

# 2. Build pipeline
pipeline = build_workplace_assistant_pipeline()

# 3. Run generation (local or Big Iron)
designer = dd.DataDesigner.from_config_builder(pipeline)
results = designer.generate(num_records=1000)
results_df = results.to_pandas()

# 4. Filter for quality
filtered_df = filter_high_quality(results_df, min_validity=4, min_complexity=3)

# 5. Save for NeMo Gym
save_for_nemo_gym(filtered_df, 'workplace_assistant_train.jsonl')
```

The output can then be used with NeMo Gym:

```bash
# Prepare data
ng_prepare_data "+config_paths=[workplace_assistant.yaml]" \
    +output_dirpath=data/workplace_assistant \
    +input_jsonl=workplace_assistant_train.jsonl

# Run GRPO training
python run_grpo_nemo_gym.py \
    --config=grpo_workplace_assistant.yaml \
    ++data.train_jsonl_fpath=data/workplace_assistant/train.jsonl
```

## Appendix A: Understanding the Reward Signal

NeMo Gym uses **state-matching** for rewards:

1. Execute the model's tool calls in a fresh environment
2. Execute the ground truth tool calls in another fresh environment
3. Compare the final database states
4. Reward = 1.0 if states match, 0.0 otherwise

This is better than action-matching because:
- Multiple valid solutions get rewarded
- Model can recover from mid-trajectory mistakes
- Focuses on goal achievement, not procedure

## Appendix B: Scaling with Big Iron

For large-scale generation, export the pipeline config and run on a cluster:

```python
from big_iron import write_config_for_bigiron, ServerConfig

server_configs = [
    ServerConfig(
        model_name="openai/gpt-4o",
        node_count=1,
        tensor_parallelism=8,
    )
]

write_config_for_bigiron(
    'workplace_assistant_sdg.json', 
    config_builder=pipeline, 
    server_configs=server_configs
)
```

Then launch:
```bash
./big-iron execute \
    --dd-config-path workplace_assistant_sdg.json \
    --num-records 10000 \
    --output-path data/workplace_assistant_synthetic
```

## Next Steps

1. **Customize prompts** for your specific domain/tools
2. **Add more patterns** to increase trajectory diversity
3. **Tune judge thresholds** based on manual inspection
4. **Iterate on quality** - check failed examples, improve prompts
5. **Scale up** using Big Iron for production data generation